## Pre-Process News Articles Dutch

In [15]:
#imports
import re
import spacy
from spacy.language import Language
from spacy.symbols import NOUN
from spacy.tokens import Doc
from spacy.util import filter_spans
import os
import pandas as pd
from tqdm import tqdm
import time
import numpy as np
import string
import nltk
from gensim.models.phrases import Phrases, Phraser

#load the Dutch model from spaCy
nlp = spacy.load("nl_core_news_lg")

#load Dutch stopwords
stop_words = spacy.lang.nl.stop_words.STOP_WORDS

In [16]:
#load the data file
df = pd.read_csv('fd_new.csv')
df


,Unnamed: 0,title,outlet,date,authors,body,word_count
0,0,95 vluchtelingen aan werk geholpen 95 vluchtel...,FD,2015-12-10 00:00:00,Unknown Authors,Amsterdam Centraal Orgaan opvang asielzoekers ...,99
1,1,Italiaanse premier krijgt wind van voren in Eu...,FD,2019-02-13 00:00:00,Unknown Authors,Brussel\nDe Italiaanse premier Giuseppe Conte ...,431
2,2,Ex-CU-leider Seegers treedt in dienst bij Otto...,FD,2023-05-26 00:00:00,Lien van der Leij,Lien van der Leij\nDen Haag\nInternationaal ar...,551
3,3,Elke inwoner is ook een consument Elke inwoner...,FD,2015-12-14 00:00:00,Unknown Authors,"[Voetnoot:] De auteur, Jan Latten, is hoofddem...",225
4,4,Er blijft voldoende werk over voor Rutte 2 Er ...,FD,2015-09-12 00:00:00,Unknown Authors,Fiscus en extra banen Uitvoering hervormingen ...,953
...,...,...,...,...,...,...,...
16544,16544,Langzaam eet Haagse politiek het lokale bestuu...,FD,2015-05-05 00:00:00,Unknown Authors,Gedachte van politici dat zij in een vergaderz...,792
16545,16545,Kwestie-Omtzigt toont kwetsbaarheid coalitie K...,FD,2017-11-14 00:00:00,Unknown Authors,De onderste steen die van Pieter Omtzigt altij...,635
16546,16546,Een land uit de EU zetten? Juridisch kan het m...,FD,2021-07-02 00:00:00,Unknown Authors,Bij de discussie over de Hongaarse lhbti-wetge...,575
16547,16547,Paniek van Chinese beleggers besmet ook beurs ...,FD,2015-07-09 00:00:00,Unknown Authors,De Hang Seng-index kent sterkste daling sinds ...,575


In [17]:
# Define a function to identify unwanted titles
def is_unwanted_title(title):
    """
    Identifies unwanted titles based on specific patterns.
    """
    unwanted_patterns = [
        "About LexisNexis", 
        "Privacy Policy", 
        "Terms & Conditions", 
        "Copyright © 2024 LexisNexis"
    ]
    # Check if any unwanted pattern is present in the title
    return any(pattern in title for pattern in unwanted_patterns)

# Filter the DataFrame to exclude unwanted rows
df_filtered = df[~df['title'].apply(is_unwanted_title)]

# Save the filtered DataFrame
df_filtered.to_csv('filtered_dataset.csv', index=False)

# Print the result
print(f"Number of rows before filtering: {len(df)}")
print(f"Number of rows after filtering: {len(df_filtered)}")

Number of rows before filtering: 16549
Number of rows after filtering: 16549


In [18]:
#drop rows with no texts
df = df.dropna(subset=['body'])

In [19]:
df['outlet'].value_counts()

outlet
FD    16549
Name: count, dtype: int64

In [20]:
df.shape

(16549, 7)

In [21]:
#sount the number of texts that exceed spaCy's limit
long_texts = df['body'].apply(lambda x: len(x) > 1000000)
num_long_texts = long_texts.sum()

print(f"Number of texts exceeding 1,000,000 characters: {num_long_texts}")


Number of texts exceeding 1,000,000 characters: 0


In [22]:
# Remove texts longer than 1,000,000 characters
df_filtered = df[df['body'].apply(len) <= 1000000]

print(f"Original dataset size: {len(df)}")
print(f"Filtered dataset size: {len(df_filtered)}")
print(f"Number of removed texts: {len(df) - len(df_filtered)}")

Original dataset size: 16549
Filtered dataset size: 16549
Number of removed texts: 0


In [23]:
def preprocess(text):
    # Remove URLs and user references; keep minimal changes for spaCy's tokenizer
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    
    # Optionally, remove extra whitespace but leave punctuation for spaCy
    text = text.lower().strip()
    
    # Process directly with spaCy (let spaCy tokenize and handle punctuation)
    doc = nlp(text)
    
    # Extract only tokens tagged as NOUN (and not stopwords or punctuation)
    nouns = [token.lemma_ for token in doc 
             if not token.is_stop 
             and not token.is_punct 
             and token.pos_ == 'NOUN'
             and len(token.lemma_) > 1]
    
    # Similarly for adjectives and verbs (if needed)
    adjectives = [token.lemma_ for token in doc if token.pos_ == "ADJ"]
    verbs = [token.lemma_ for token in doc if token.pos_ == "VERB"]

    # Combine all tokens into one processed string
    processed = ' '.join(nouns + adjectives + verbs)

    return processed, ' '.join(nouns), ' '.join(adjectives), ' '.join(verbs)

In [24]:
from tqdm import tqdm
tqdm.pandas()

# Unpack the tuple into separate columns
df_filtered[['processed', 'nouns', 'adjectives', 'verbs']] = df_filtered['body'].progress_apply(preprocess).tolist()

  0%|          | 0/16549 [00:00<?, ?it/s]

100%|██████████| 16549/16549 [42:00<00:00,  6.57it/s]  


In [25]:
#inspect df
df_filtered.head()

,Unnamed: 0,title,outlet,date,authors,body,word_count,processed,nouns,adjectives,verbs
0,0,95 vluchtelingen aan werk geholpen 95 vluchtel...,FD,2015-12-10 00:00:00,Unknown Authors,Amsterdam Centraal Orgaan opvang asielzoekers ...,99,orgaan opvang asielzoeker coa uitzendorganisat...,orgaan opvang asielzoeker coa uitzendorganisat...,centraal,starten begeleiden willen verblijven begeleide...
1,1,Italiaanse premier krijgt wind van voren in Eu...,FD,2019-02-13 00:00:00,Unknown Authors,Brussel\nDe Italiaanse premier Giuseppe Conte ...,431,premier wind parlement straatsburg europarleme...,premier wind parlement straatsburg europarleme...,italiaans stevig Europees Italiaans Frans geel...,krijgen voelen ïnstalleeren aflopen liggen wei...
2,2,Ex-CU-leider Seegers treedt in dienst bij Otto...,FD,2023-05-26 00:00:00,Lien van der Leij,Lien van der Leij\nDen Haag\nInternationaal ar...,551,arbeidsbemiddelaar organisatie uitzendbureau a...,arbeidsbemiddelaar organisatie uitzendbureau a...,internationaal strategisch nieuw duurzaam circ...,hebben aantrekken aflopen afzwaaien starten ga...
3,3,Elke inwoner is ook een consument Elke inwoner...,FD,2015-12-14 00:00:00,Unknown Authors,"[Voetnoot:] De auteur, Jan Latten, is hoofddem...",225,voetnoot auteur latt hoogleraar cbsprognose te...,voetnoot auteur latt hoogleraar cbsprognose te...,Hoofddemograaf voorlopig centraal recentelijk ...,laten zien doorgroeien bereiken presenteren ve...
4,4,Er blijft voldoende werk over voor Rutte 2 Er ...,FD,2015-09-12 00:00:00,Unknown Authors,Fiscus en extra banen Uitvoering hervormingen ...,953,fiscus baan uitvoering hervorming gas energie ...,fiscus baan uitvoering hervorming gas energie ...,extra groen drie kort impopulair grootscheeps ...,gaan regeren presenteren volgen gaan doorvoere...


In [26]:
#df_filtered.to_pickle('processed.pkl')
#df_filtered.to_csv('processed.csv')

#df_filtered.to_pickle('vk_processed.pkl')
#df_filtered.to_csv('vk_processed.csv')
#df_filtered.to_csv('TR_online_processed.csv')
df_filtered.to_csv('FD_processed.csv')

In [27]:
df_filtered.shape

(16549, 11)